In [10]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [11]:
import logging
import pandas as pd

from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [12]:
customer = "MissPompadour GmbH"
source = "googleAdwords"
url = return_api_url()
url = "https://api.innkeepr.ai/api"
print(f"url = {url}")
list_workspaces = return_workspace_ids()
workspace_id = [acc["id"] for acc in list_workspaces if acc["name"] == customer]
if len(workspace_id) != 1:
    print(sorted([account["name"] for account in list_workspaces]))
account_id = workspace_id[0]

In [13]:
df = pd.read_csv("DataChecks/conversion_treatment_match/signal_treatment_conversion_match_raw.csv")
df

In [38]:
conv = df[df["conv_name"]=="order completed"]
treatments = ['69deca5887320b6c0f1461a8',
  '6a30875887320b66b46d5e3b']
conv = conv[conv["treatment"].isin(treatments)]
print(conv["session_count"].sum())
conv

In [23]:
treatments = conv["treatment"].dropna().unique().tolist()
len(treatments)

In [18]:
treatments_api = send_to_innkeepr_api_paginated(
    f"{url}/treatments/query",
    account_id,
    {"id":treatments},
    logging
)
treatments_api = pd.json_normalize(treatments_api)
treatments_api.head()

In [26]:
treatments_api_google = treatments_api[treatments_api["connection"]=="609ffc4578188e83a2bc2c2c"]
treatments_api_google[["id","name","connection","relates_to.campaign.name","externalId"]]

In [27]:
treatments_api_google["relates_to.campaign.name"].value_counts()

In [ ]:
#6490552367
#ohne zuordnung gclid="EAIaIQobChMIgN3XiP36lAMVbZpoCR0RqxDlEAQYDCABEgIvdPD_BwE"
#gclid mit zuordnung: gclid = "CjwKCAjw857RBhAgEiwAI-1yKP0TbgteL82aYAqar6qmOTGOspOjMVKQDmcY4F6ogj3QRMPbIpIHcRoCfVYQAvD_BwE"
external_ids = send_to_innkeepr_api_paginated(
    f"{url}/ads/query",
    account_id,
    {"externalId":gclid},
    logging
)
external_ids = pd.json_normalize(external_ids)
external_ids.head()